### loss function (손실함수)
: 모델의 예측값이 실제 정답과 얼마나 다른지 수치로 계산하는 함수<br>
모델의 목표는 손실값을 줄이는 것
- 기본용어
    - prediction: 모델이 예측한 값
    - target: 정답
    - loss: 예측값과 정답의 차이를 계산한 값
- 손실함수의 역할
    - 예측값과 정답의 차이를 수치로 표현
    - 모델이 얼마나 잘 예측했는지 평가할 수 있음
    - 자동 미분을 통해 weight와 bias를 업데이트 할 때 사용할 기울기를 계산할 수 있음
#### MSELoss (mean squared error)
: 평균 제곱 오차를 계산하는 손실함수<br>
예측값과 정답의 차이를 제곱한 뒤 평균 계산
- 특징
    - 오차를 제곱하므로 큰 오차에 더 큰 패널티를 부여함
    - 손실값이 0이상임
    - 예측값이 정답과 같으면 손실값은 0
    - 주로 회귀문제에서 사용
- 사용방법<br>
&emsp;lossfun=nn.MSELoss()<br>
&emsp;loss=lossfun(prediction, target)<br>
기본 설정 reduction='mean'은 모든 원소의 제곱오차의 평균

In [3]:
import torch
import torch.nn as nn

In [ ]:
prediction = torch.tensor([2.0, 4, 6])
target = torch.tensor([1.0, 5, 4])
lf = nn.MSELoss()
loss = lf(prediction, target)
print(prediction, target, loss)
loss2 = ((target - prediction) ** 2).mean()
print(loss2)

tensor([2., 4., 6.]) tensor([1., 5., 4.]) tensor(2.)
tensor(2.)


#### reduction 옵션
|옵션|의미|
|:---:|:---:|
|mean|모든 원소의 손실 평균(기본값)|
|sum|모든 원소의 손실 합|
|none|원소별 손실을 그대로 반환|

**none은 데이터별 손실을 따로 확인하거나 가중치를 적용할 때 유용

In [23]:
prediction = torch.tensor([2.0, 4, 6])
target = torch.tensor([1.0, 5, 4])
loss_mean = nn.MSELoss(reduction="mean")(prediction, target)
loss_sum = nn.MSELoss(reduction="sum")(prediction, target)
loss_none = nn.MSELoss(reduction="none")(prediction, target)
print(loss_mean, loss_sum, loss_none)

tensor(2.) tensor(6.) tensor([1., 1., 4.])


#### L1Loss(mean absolute error)
: 예측값과 정답의 차이를 절댓값을 적용한 뒤 평균 계산
- 특징
    - 오차에 절댓값 사용
    - 손실값이 0 이상
    - 예측값과 정답이 같으면 손실값은 0
    - MSE보다 큰 오차에 대한 패널티가 상대적으로 작음
    - 주로 회귀 문제에서 사용
- 사용법
    - loss_fn=nn.L1Loss()
    - loss=loss_fn(prediction, target)

In [10]:
prediction = torch.tensor([2, 4, 6.0])
target = torch.tensor([1.0, 5, 4])
lf = nn.L1Loss()
loss = lf(prediction, target)
print(prediction, target, loss)
loss2 = torch.abs(target - prediction).mean()
print(loss2)

tensor([2., 4., 6.]) tensor([1., 5., 4.]) tensor(1.3333)
tensor(1.3333)


- 큰 오차를 더 강하게 반영하고 싶다면 MSE를, 큰 오차나 이상치의 영향을 상대적으로 적게 받고 싶으면 L1

In [11]:
pr = torch.tensor([10.0, 10, 10])
tg = torch.tensor([9.0, 8, 0])
ml = nn.MSELoss()(pr, tg)
l1lo = nn.L1Loss()(pr, tg)
print(ml, l1lo)

tensor(35.) tensor(4.3333)


#### BCELoss(binary cross entropy)
: 이진분류에서 사용하는 손실함수<br>
**중요**<br>
모델이 sigmoid를 통과한 확률값을 입력으로 받아야함<br>
nn.BCELoss()(prebability, target)<br>
&emsp;prebability: 0~1사이의 확률값<br>
&emsp;target: 정답. 일반적으로 0또는 1이며 실수형 텐서로 전달
- 실제 정답이 1인데 확률이 1에 가까우면 손실이 작아지고 0에 가까우면 손실이 커짐
- 극단적인 값에서 NaN 발생 위험 있음

In [12]:
prob = torch.tensor([0.9, 0.2, 0.7, 0.1])
tg = torch.tensor([1.0, 0, 1, 0])
loss = nn.BCELoss()(prob, tg)
print(loss)

tensor(0.1976)


#### BCEWithLogitLoss
: sigmoid와 BCELoss 계산을 하나로 결합한 손실함수
- 중요한 차이
    - BCELoss(): sigmoid를 적용한 확률값을 입력
    - BCEWithLogitsLoss(): sigmoid를 적용하지 않은 logits을 입력으로 받음
- 사용법
    - nn.BCEWithLogitsLoss()(logits, target)
- 왜 사용? sigmoid와 bceloss계산을 결합하여 수치적으로 더 안정적인 방식으로 손실을 계산할 수 있음, 내부적으로 안정적인 연산 수행(극단적인 값에서 NaN 발생 위험이 적음)

In [16]:
lg = torch.tensor([2.0, -1.5, 1, -2])
tg = torch.tensor([1.0, 0, 1, 0])
lo = nn.BCEWithLogitsLoss()(lg, tg)
print(lg, lo)

tensor([ 2.0000, -1.5000,  1.0000, -2.0000]) tensor(0.1921)


In [19]:
lg = torch.tensor([1.2, -0.7, 2, -1.5])
tg = torch.tensor([1.0, 0, 1, 0])
prob = torch.sigmoid(lg)
lb = nn.BCELoss()(prob, tg)
ll = nn.BCEWithLogitsLoss()(lg, tg)
print(lb)
print(ll)
print(torch.allclose(lb, ll))

tensor(0.2487)
tensor(0.2487)
True


In [ ]:
logi_a = torch.tensor([1000.0], requires_grad=True)
tg = torch.tensor([0.0])
pb = torch.sigmoid(logi_a)
la = nn.BCELoss()(pb, tg)
la.backward()
logi_b = torch.tensor([1000.0])
lb = nn.BCEWithLogitsLoss()(logi_b, tg)

In [20]:
# Loss 값이 차이나고, 기울기가 소실되는 예
tg = torch.tensor([0.0])
la = torch.tensor([1000.0], requires_grad=True)
pb = torch.sigmoid(la)
lossa = nn.BCELoss()(pb, tg)
la.backward()
lb = torch.tensor([1000.0], requires_grad=True)
lossb = nn.BCEWithLogitsLoss()(lb, tg)
lb.backward()
print("sigmoid확률:", pb.item())
print("bcewith~:", lb.item())
print("bceloss gradient:", la.grad.item())
print("bcewiht~ gradient:", lb.grad.item())

sigmoid확률: 1.0
bcewith~: 1000.0
bceloss gradient: 1.0
bcewiht~ gradient: 1.0


#### CrossEntropyLoss
: 여러 클래스 중 하나를 선택하는 다중 분류에서 주로 사용
- 사용방법
    - nn.CrossEntropyLoss()(logits, target)
- 입력형식
    - 모델 출력(logits): (batch_size, num_classes)
    - 정답(target): (batch_size)
    - 정답의 자료형: torch.long
- 내부적으로 LogSoftmax와 NLLLoss에 해당하는 계산을 수행
    - LogSoftmax: softmax결과에 log를 적용한 함수
    - NLLLoss: 그 로그 확률을 이용하여 손실을 계산하는 함수

In [21]:
lg = torch.tensor([[2.0, 1, 0.1, 0.5], [0.2, 2.5, 1, 0.3], [0.1, 0.4, 0.8, 2]])
tg = torch.tensor([0, 1, 3.0], dtype=torch.long)
loss = nn.CrossEntropyLoss()(lg, tg)
print(loss)

tensor(0.4724)


#### 문제 유형에 따른 손실함수 선택
|문제 유형|대표 손실함수|모델 선택|
|:---:|:---:|:---:|
|회귀|MSELoss, L1Loss|일반적인 실수값|
|이진분류|BCELoss|Sigmoid 적용 확률|
|이진분류|BCEWithLogitsLoss|Sigmoid 적용 전 logits|
|다중분류|CrossEntropyLoss|Sigmoid 적용 전 logits|

In [22]:
# 각 손실함수의 입력 형태에 따른 확인
# 회귀
regression_prediction = torch.tensor([2.5, 3])
regression_target = torch.tensor([3.0, 2])
print("MSE:", nn.MSELoss()(regression_prediction, regression_target).item())
print("L1:", nn.L1Loss()(regression_prediction, regression_target).item())

# 이진분류
binary_logits = torch.tensor([1.0, -1])
binary_target = torch.tensor([1.0, 0])
print("bcewith~:", nn.BCEWithLogitsLoss()(binary_logits, binary_target).item())

# 다중분류
multi_logits = torch.tensor([[1.0, 2, 0.5], [2.0, 0.5, 1]])
multi_target = torch.tensor([1.0, 0], dtype=torch.long)
print("creossentropy~:", nn.CrossEntropyLoss()(multi_logits, multi_target).item())

MSE: 0.625
L1: 0.75
bcewith~: 0.3132617175579071
creossentropy~: 0.4643687903881073
